# 复现 CALM 论文 Table 7 (Appendix C: 无数据 standardization)

**目标实验** ：CALM (kaifeng-jin) 论文 Appendix C, Table 7
- 50-node ER1, 50-node ER4, 100-node ER1
- 不做 data standardization
- 比较 CALM / GOLEM-NV-l1 / NOTEARS / PC / FGES
- 指标：CPDAG-SHD, Skeleton Precision, Skeleton Recall (mean ± std)

**本 notebook 的修改 (用户要求)**
- 样本量统一 `n = 20000` (不用论文的 1000 与 10⁶)
- `noise_ratio = 16` (CalmDataset, mode=variance；非等方差)
- 额外加入两个算法：`cd_A`、`cd_A_weakfaith`
- trials = 5

数据生成全部使用 `calm_dataset.CalmDataset` (与 CALM 论文同源)。

## 1. 环境与导入

In [1]:
import logging
import os
import sys
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    display = print


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for path in [start, *start.parents]:
        if (path / 'calm_dataset.py').exists() and (path / 'coordinate_descent').exists():
            return path
    raise RuntimeError(f'Could not find repo root from {start}')


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
if not os.environ.get('JAVA_HOME'):
    _jdk_candidate = r'C:\Program Files\Microsoft\jdk-21.0.10.7-hotspot'
    if os.path.isdir(_jdk_candidate):
        os.environ['JAVA_HOME'] = _jdk_candidate

from MEC import find_v_structures, get_skeleton, is_in_markov_equiv_class
from calm_dataset import CalmDataset, weight_to_binary_adj
from coordinate_descent.coordinate0 import dag_coordinate_descent_l0
from coordinate_descent.cd_A_weakfaith import dag_coordinate_descent_l0_weakfaith

print('Python   :', sys.version.split()[0])
print('Repo root:', REPO_ROOT)
print('cd_A          : OK')
print('cd_A_weakfaith: OK')

/home/yin/DAG/experiments/notebooks/test/2026-05-13
Python   : 3.10.12
Repo root: /home/yin/DAG
cd_A          : OK
cd_A_weakfaith: OK


In [2]:
# ── CALM (kaifeng-jin) ────────────────────────────────────────
CALM_REPO_CANDIDATES = [
    Path(os.environ['CALM_REPO']) if os.environ.get('CALM_REPO') else None,
    REPO_ROOT / 'external' / 'CALM',
    REPO_ROOT.parent / 'CALM',
    Path(r'D:\tmp\CALM-inspect'),
]
CALM_REPO = next(
    (p.resolve() for p in CALM_REPO_CANDIDATES
     if p is not None and (p / 'CALM.py').exists()),
    None,
)
CALM_IMPORT_ERROR = None
try:
    if CALM_REPO is None:
        raise FileNotFoundError('CALM repo not found; set $CALM_REPO or clone to external/CALM')
    if str(CALM_REPO) not in sys.path:
        sys.path.insert(0, str(CALM_REPO))
    import torch
    from CALM import calm as calm_algorithm
    HAS_CALM = True
    print(f'CALM      : OK ({CALM_REPO})')
except Exception as exc:
    HAS_CALM = False
    CALM_IMPORT_ERROR = exc
    print(f'CALM unavailable: {exc}')

# ── GOLEM-NV-l1 ───────────────────────────────────────────────
GOLEM_IMPORT_ERROR = None
try:
    _golem_src = str((REPO_ROOT / 'golemMain' / 'src').resolve())
    if _golem_src not in sys.path:
        sys.path.append(_golem_src)
    from golem import golem as golem_fit
    HAS_GOLEM = True
    print('GOLEM     : OK')
except Exception as exc:
    HAS_GOLEM = False
    GOLEM_IMPORT_ERROR = exc
    print(f'GOLEM unavailable: {exc}')

# ── NOTEARS (gcastle) ─────────────────────────────────────────
NOTEARS_IMPORT_ERROR = None
_prev_disable = logging.root.manager.disable
logging.disable(logging.INFO)
try:
    from castle.algorithms import Notears as _Notears
    HAS_NOTEARS = True
    print('NOTEARS   : OK')
except Exception as exc:
    HAS_NOTEARS = False
    NOTEARS_IMPORT_ERROR = exc
    print(f'NOTEARS unavailable: {exc}')
finally:
    logging.disable(_prev_disable)

# ── Tetrad: FGES & PC (需要 JDK 21+) ────────────────────────────
TETRAD_IMPORT_ERROR = None
try:
    import fges_compat as _tetrad_mod
    _probe = pd.DataFrame(np.eye(2), columns=['x0', 'x1'])
    _s = _tetrad_mod.TetradSearch(_probe)
    _s.set_verbose(False)
    _s.use_sem_bic()
    _s.run_fges()
    del _probe, _s
    HAS_TETRAD = True
    print('Tetrad    : OK (FGES + PC available)')
except Exception as exc:
    HAS_TETRAD = False
    TETRAD_IMPORT_ERROR = exc
    print(f'Tetrad unavailable: {exc}')

# ── CPDAG-SHD: causaldag (★ paper 用的库) > cdt > 手写 fallback ─────
CAUSALDAG_IMPORT_ERROR = None
try:
    import causaldag as _causaldag
    import igraph as _ig_for_shd
    HAS_CAUSALDAG = True
    print('causaldag : OK (★ 与论文 compute_shd_cpdag 同一实现)')
except Exception as exc:
    HAS_CAUSALDAG = False
    CAUSALDAG_IMPORT_ERROR = exc
    print(f'causaldag : MISSING ({exc})')

try:
    _toolbox = str((REPO_ROOT / 'toolbox').resolve())
    if _toolbox not in sys.path:
        sys.path.append(_toolbox)
    from cdt.metrics import SHD_CPDAG as _SHD_CPDAG
    HAS_CPDAG_SHD = True
    print('cdt.SHD_CPDAG: OK (secondary backend)')
except Exception as exc:
    _SHD_CPDAG = None
    HAS_CPDAG_SHD = False
    print(f'cdt.SHD_CPDAG: 不可用 ({exc})')

if HAS_CAUSALDAG:
    print(f'\n★ CPDAG-SHD 主路径: causaldag (与论文一致)')
elif HAS_CPDAG_SHD:
    print(f'\n⚠ CPDAG-SHD 主路径: cdt.SHD_CPDAG (近似但与论文实现不同)')
else:
    print(f'\n⚠ CPDAG-SHD 主路径: 手写 skel+v-struct 近似 (会系统性偏大 ~2.4×)')

CALM      : OK (/home/yin/DAG/external/CALM)


2026-05-15 10:10:35.286282: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-15 10:10:35.392585: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-15 10:10:35.392623: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-15 10:10:35.393948: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-15 10:10:35.417616: I tensorflow/core/platform/cpu_feature_guar

2026-05-15 10:10:36.826388: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


GOLEM     : OK
NOTEARS   : OK


May 15, 2026 10:10:38 AM java.util.prefs.FileSystemPreferences$6 run


Tetrad    : OK (FGES + PC available)


/home/yin/DAG/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


causaldag : OK (★ 与论文 compute_shd_cpdag 同一实现)


cdt.SHD_CPDAG: OK (secondary backend)

★ CPDAG-SHD 主路径: causaldag (与论文一致)


Detecting 2 CUDA device(s).


## 2. 实验配置 (复现 Table 7，含用户修改)

In [3]:
CFG = {
    # ── 实验规模 ──────────────────────────────────────────
    'trials': 5,
    'seed':   42,

    # 论文 Table 7 的三个 graph 配置
    # CalmDataset: s0 = round(degree * d) 条边 → ER{k} 即 calm_degree=k
    'graph_configs': [
        {'label': '50-node ER1',  'd': 50,  'calm_degree': 1.0, 'graph_type': 'ER'},
        {'label': '50-node ER2',  'd': 50,  'calm_degree': 2.0, 'graph_type': 'ER'},
        {'label': '50-node ER4',  'd': 50,  'calm_degree': 4.0, 'graph_type': 'ER'},
        # {'label': '100-node ER1', 'd': 100, 'calm_degree': 1.0, 'graph_type': 'ER'},
    ],

    # 样本量 (用户要求统一 20000，而非论文的 1000/1e6)
    'n_samples': 50000,

    # ── 数据生成 (CalmDataset) ────────────────────────────
    'sem_type':         'gauss',
    'noise_ratio':      16.0,           # 用户要求；非等方差
    'noise_scale_mode': 'variance',
    'b_scale':          1.0,
    'standardize':      False,          # ★ Table 7 关键: 不 standardize

    # ── CALM ──────────────────────────────────────────────
    'calm_lambda1':         0.005,
    'calm_alpha':           0.01,
    'calm_tau':             0.5,
    'calm_rho_init':        1e-5,
    'calm_rho_mult':        3.0,
    'calm_htol':            1e-8,
    'calm_subproblem_iter': 10000,
    'calm_standardize':     False,      # ★ paper Table 7
    'calm_device':          'cpu',

    # ── GOLEM-NV-l1 (论文默认) ─────────────────────────────
    # 见 golemMain/src/golem.py docstring:
    #   GOLEM-NV: equal_variances=False, lambda_1=2e-3, lambda_2=5.0
    'golem_equal_variances': False,
    'golem_lambda1':         2e-3,
    'golem_lambda2':         5.0,
    'golem_num_iter':        10000,    # 论文 1e5
    'golem_learning_rate':   1e-3,
    'golem_threshold':       0.3,       # 作用在 GOLEM 学到的边权矩阵 B 上

    # ── NOTEARS ───────────────────────────────────────────
    'notears_lambda1':   0.1,
    'notears_loss_type': 'l2',
    'notears_threshold': 0.3,           # 作用在 NOTEARS 学到的边权矩阵 W 上
    # ★ runtime tuning: 默认 h_tol=1e-8 + max_iter=100 在稠密图(noise_ratio=16)上极慢
    'notears_h_tol':     1e-4,          # 1e-8 -> 1e-4 (二值图几乎不变,主要 runtime killer)
    'notears_max_iter':  20,            # 100  -> 20  (兜底 AL 外层 iter)

    # ── PC (Tetrad, FisherZ) ───────────────────────────────
    'pc_alpha':  0.01,
    'pc_stable': True,
    'pc_depth':  -1,

    # ── FGES (Tetrad, SEM-BIC) ─────────────────────────────
    'fges_penalty_discount': 1.0,

    # ── cd_A / cd_A_weakfaith ──────────────────────────────
    # cd_threshold 作用在 cd 学到的权重矩阵 A 上 (语义 ≠ GOLEM/NOTEARS):
    #   G[i,j] = 1 iff |A[i,j]| > cd_threshold
    # 项目其它实验全部用 0.05 (= coordinate0.py 默认值)，沿用约定
    'cd_T':         1000000,
    'cd_threshold': 0.05,
    'cd_lambda_l0': 0.1,
    'wf_tau':       0.05,
    'wf_screening': 'pcorr',
    'wf_combine':   'union',
    'wf_sampling_mode': 'preserve',

    # ── 输出 ──────────────────────────────────────────────
    'out_dir': str((REPO_ROOT / 'experiments' / 'results').resolve()),
    'tag':     'calm_table7_repro',
}
os.makedirs(CFG['out_dir'], exist_ok=True)

# 算法顺序：与论文 Table 7 一致 (论文 5 个) + 用户加的 cd_A 系列
ALGORITHM_ORDER = [
    'CALM',
    'GOLEM-NV-l1',
    'NOTEARS',
    'PC',
    'FGES',
    'cd_A',
    'cd_A_weakfaith',
]

print(f"trials       : {CFG['trials']}")
print(f"n_samples    : {CFG['n_samples']}")
print(f"noise_ratio  : {CFG['noise_ratio']} ({CFG['noise_scale_mode']})")
print(f"standardize  : {CFG['standardize']}")
print('graph configs:')
for gc in CFG['graph_configs']:
    s0 = int(round(gc['calm_degree'] * gc['d']))
    print(f"  {gc['label']:14s}  d={gc['d']:3d}  s0={s0:4d} edges")
print('algorithms   :', ALGORITHM_ORDER)

trials       : 5
n_samples    : 50000
noise_ratio  : 16.0 (variance)
standardize  : False
graph configs:
  50-node ER1     d= 50  s0=  50 edges
  50-node ER2     d= 50  s0= 100 edges
  50-node ER4     d= 50  s0= 200 edges
algorithms   : ['CALM', 'GOLEM-NV-l1', 'NOTEARS', 'PC', 'FGES', 'cd_A', 'cd_A_weakfaith']


## 3. 评估函数

Table 7 用的三个指标 (CALM 论文一致定义):
- **SHD of CPDAG**：估计 CPDAG 与真实 CPDAG 的 SHD
- **Precision of Skeleton**：在无向骨架上 TP / (TP + FP)
- **Recall    of Skeleton**：在无向骨架上 TP / (TP + FN)

**额外追加**：
- **SHD (directed)**：有向边上的结构汉明距离 (DAG 估计专用)
  - PC、FGES 输出 CPDAG（含未定向边的双向编码），与有向真值算 SHD 没意义；因此 **SHD 在 PC/FGES 上跳过 (NaN)**。

### CPDAG-SHD 实现 (与论文 `compute_shd_cpdag` 对齐)

所有指标统一从仓库根目录的 [`dag_metrics.py`](../../../dag_metrics.py) 导入，避免每个 notebook 重复实现导致漂移。

`cpdag_shd` 与 paper 的 `metrics.compute_shd_cpdag` 等价：
```python
import causaldag as cd
cpdag_true = cd.DAG.from_amat(B_true).cpdag().to_amat()[0]
cpdag_est  = cd.DAG.from_amat(B_est).cpdag().to_amat()[0]   # paper 要求 est 是 DAG
return cd.PDAG.from_amat(cpdag_true).shd(cd.PDAG.from_amat(cpdag_est))
```

`dag_metrics.cpdag_shd` 在此基础上把 "est 必须是 DAG" 放宽：
- 若 G_est 是 DAG（NOTEARS / GOLEM / CALM / cd_A 系列）→ 走 `DAG.cpdag()`
- 若 G_est 是 PDAG/CPDAG（PC / FGES）→ 直接 `cd.PDAG.from_amat(G_est)`

后端优先级：`causaldag > cdt.SHD_CPDAG > 手写 skel+v-struct 近似`（最后一档已知偏大 ~2.4×，会发 RuntimeWarning）。

In [4]:
# 全部 DAG/CPDAG 指标改从 dag_metrics 导入 (paper-aligned causaldag 路径)。
# 后续所有 notebook 都应该从这里导入，避免实现漂移。
from dag_metrics import (
    cpdag_shd,
    shd_directed,
    skeleton_precision_recall,
    evaluate,
    is_dag as _is_dag_int,
    get_cpdag_shd_backend,
    SHD_SKIP_ALGORITHMS,
)


def tetrad_matrix_to_adj(df_result) -> np.ndarray:
    """Tetrad endpoint matrix -> 0/1 adjacency (ARROW=2, TAIL=3).

    Undirected edges encoded bidirectionally (G[i,j]=G[j,i]=1) so cpdag_shd
    can route them through cd.PDAG.from_amat correctly.
    """
    mat = df_result.values if hasattr(df_result, 'values') else df_result
    d   = mat.shape[0]
    G   = np.zeros((d, d), dtype=int)
    for i in range(d):
        for j in range(i + 1, d):
            a, b = int(mat[i, j]), int(mat[j, i])
            if   a == 2 and b == 3: G[i, j] = 1
            elif a == 3 and b == 2: G[j, i] = 1
            elif a == 3 and b == 3: G[i, j] = G[j, i] = 1
            elif a != 0 or  b != 0: G[i, j] = G[j, i] = 1
    np.fill_diagonal(G, 0)
    return G


def make_dataset(d: int, calm_degree: float, graph_type: str, n: int, seed: int):
    ds = CalmDataset(
        n=n, d=d,
        graph_type=graph_type, degree=calm_degree,
        sem_type=CFG['sem_type'],
        seed=int(seed),
        noise_ratio=CFG['noise_ratio'],
        noise_scale_mode=CFG['noise_scale_mode'],
        b_scale=CFG['b_scale'],
    )
    G_true = weight_to_binary_adj(ds.B)
    return ds.X, G_true


print('评估函数已就绪 (从 dag_metrics 导入)')
print(f'  CPDAG-SHD 后端       : {get_cpdag_shd_backend()}')
print(f'  有向 SHD 跳过的算法   : {sorted(SHD_SKIP_ALGORITHMS)}')

评估函数已就绪 (从 dag_metrics 导入)
  CPDAG-SHD 后端       : causaldag
  有向 SHD 跳过的算法   : ['FGES', 'PC']


## 4. 算法封装 (7 个)

In [5]:
def _failure(msg) -> dict:
    return {'status': 'failed', 'G_est': None, 'message': str(msg)}


def _sample_cov(X: np.ndarray) -> np.ndarray:
    Xc = X - X.mean(axis=0, keepdims=True)
    return Xc.T @ Xc / Xc.shape[0]


def run_CALM(X, seed):
    if not HAS_CALM:
        return _failure(f'CALM unavailable: {CALM_IMPORT_ERROR}')
    np.random.seed(seed)
    torch.manual_seed(seed)
    B_w = calm_algorithm(
        X,
        lambda1=CFG['calm_lambda1'],
        alpha=CFG['calm_alpha'],
        tau=CFG['calm_tau'],
        rho_init=CFG['calm_rho_init'],
        rho_mult=CFG['calm_rho_mult'],
        htol=CFG['calm_htol'],
        subproblem_iter=CFG['calm_subproblem_iter'],
        standardize=CFG['calm_standardize'],
        device=CFG['calm_device'],
    )
    G_est = (np.abs(np.asarray(B_w)) > 0).astype(int)
    np.fill_diagonal(G_est, 0)
    return {'status': 'ok', 'G_est': G_est, 'message': ''}


def run_GOLEM_NV_l1(X, seed):
    if not HAS_GOLEM:
        return _failure(f'GOLEM unavailable: {GOLEM_IMPORT_ERROR}')
    B_est = golem_fit(
        X,
        lambda_1=CFG['golem_lambda1'],
        lambda_2=CFG['golem_lambda2'],
        equal_variances=CFG['golem_equal_variances'],
        num_iter=CFG['golem_num_iter'],
        learning_rate=CFG['golem_learning_rate'],
        seed=int(seed),
    )
    G_est = (np.abs(np.asarray(B_est)) > CFG['golem_threshold']).astype(int)
    np.fill_diagonal(G_est, 0)
    return {'status': 'ok', 'G_est': G_est, 'message': ''}


def run_NOTEARS(X, seed):
    if not HAS_NOTEARS:
        return _failure(f'NOTEARS unavailable: {NOTEARS_IMPORT_ERROR}')
    model = _Notears(
        lambda1=CFG['notears_lambda1'],
        loss_type=CFG['notears_loss_type'],
        w_threshold=CFG['notears_threshold'],
        h_tol=CFG['notears_h_tol'],
        max_iter=CFG['notears_max_iter'],
    )
    _prev = logging.root.manager.disable
    logging.disable(logging.INFO)
    try:
        model.learn(X)
    finally:
        logging.disable(_prev)
    G_est = np.asarray(model.causal_matrix, dtype=int)
    np.fill_diagonal(G_est, 0)
    return {'status': 'ok', 'G_est': G_est, 'message': ''}


def run_PC(X, seed):
    if not HAS_TETRAD:
        return _failure(f'Tetrad unavailable: {TETRAD_IMPORT_ERROR}')
    cols = [f'x{i}' for i in range(X.shape[1])]
    df_X = pd.DataFrame(X, columns=cols).astype('float64')
    search = _tetrad_mod.TetradSearch(df_X)
    search.set_verbose(False)
    search.run_pc(
        alpha=CFG['pc_alpha'],
        stable=CFG['pc_stable'],
        depth=CFG['pc_depth'],
    )
    G_est = tetrad_matrix_to_adj(search.get_graph_to_matrix())
    return {'status': 'ok', 'G_est': G_est, 'message': ''}


def run_FGES(X, seed):
    if not HAS_TETRAD:
        return _failure(f'Tetrad unavailable: {TETRAD_IMPORT_ERROR}')
    cols = [f'x{i}' for i in range(X.shape[1])]
    df_X = pd.DataFrame(X, columns=cols).astype('float64')
    search = _tetrad_mod.TetradSearch(df_X)
    search.set_verbose(False)
    search.use_sem_bic(penalty_discount=CFG['fges_penalty_discount'])
    search.run_fges()
    G_est = tetrad_matrix_to_adj(search.get_graph_to_matrix())
    return {'status': 'ok', 'G_est': G_est, 'message': ''}


def run_cd_A(X, seed):
    S = _sample_cov(X)
    A, G_est, obj = dag_coordinate_descent_l0(
        S=S, T=CFG['cd_T'], seed=int(seed),
        threshold=CFG['cd_threshold'], lambda_l0=CFG['cd_lambda_l0'],
    )
    return {'status': 'ok', 'G_est': G_est, 'message': ''}


def run_cd_A_weakfaith(X, seed):
    S = _sample_cov(X)
    A, G_est, obj = dag_coordinate_descent_l0_weakfaith(
        S=S, T=CFG['cd_T'], seed=int(seed),
        threshold=CFG['cd_threshold'], lambda_l0=CFG['cd_lambda_l0'],
        faithfulness_tau=CFG['wf_tau'],
        screening=CFG['wf_screening'],
        combine=CFG['wf_combine'],
        sampling_mode=CFG['wf_sampling_mode'],
    )
    return {'status': 'ok', 'G_est': G_est, 'message': ''}


RUNNERS = {
    'CALM':           run_CALM,
    'GOLEM-NV-l1':    run_GOLEM_NV_l1,
    'NOTEARS':        run_NOTEARS,
    'PC':             run_PC,
    'FGES':           run_FGES,
    'cd_A':           run_cd_A,
    'cd_A_weakfaith': run_cd_A_weakfaith,
}
RUNNERS = {k: RUNNERS[k] for k in ALGORITHM_ORDER if k in RUNNERS}
list(RUNNERS)

['CALM', 'GOLEM-NV-l1', 'NOTEARS', 'PC', 'FGES', 'cd_A', 'cd_A_weakfaith']